# Maia Corpus v9 — Gutenberg Full Acquisition + Automated Rights v0.8

**Produção, sem novo piloto.** Aquisição integral das partições `pt`, `en`, `es` de `manu/project_gutenberg` por streaming, retomada persistente, logs ao vivo, deduplicação e auditoria de direitos automatizada. Reaproveita os 30 livros v0.6. **Os originais permanecem em quarentena; nenhum documento é liberado por ausência de evidência.**

**Como executar:** execute as células 1–5. A célula 5 pode levar muitas horas e ser interrompida: execute-a novamente para continuar. Após a aquisição, execute as células 6–8. O processamento é incremental e não exige terminar todos os idiomas para produzir relatórios. **Verifique espaço disponível no Drive:** o conjunto pode consumir dezenas de GB; o notebook não pode aumentar a cota da conta.

**Retomada:** o estado guarda o índice de leitura da partição, os IDs e hashes já registrados em manifesto e os arquivos escritos atomicamente. A retomada pode reabrir/percorrer parcialmente o stream (custo de rede), mas não baixa novamente livros íntegros já persistidos. Se a sessão do Colab cair, os arquivos permanecem no Drive.

## 1. Montagem do Drive e diretórios persistentes

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, sys, subprocess, datetime, time, signal, shutil

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
BASE=ROOT/'gutenberg_v08'
OLD=ROOT/'gutenberg_v06'
for name in ['quarantine/books','manifests','state','reports','logs','workers','configs','rights','cleaned_quarantine']:
    (BASE/name).mkdir(parents=True,exist_ok=True)
assert (ROOT/'gutenberg_v06'/'manifests').exists(), 'Não encontrei os manifestos v0.6'
print('Destino persistente:',BASE)
print('Espaço livre no runtime (não é a cota do Drive):',shutil.disk_usage('/content').free//(1024**3),'GiB')

## 2. Configuração de produção

`max_unique_per_language: null` significa **sem limite de livros**. `max_scan_per_language: null` significa percorrer a partição inteira. A lista `languages` define a ordem. `min_free_drive_bytes` é uma verificação local indicativa (em Drive montado, pode não refletir a cota remota); erros de cota serão capturados e permitirão retomada após liberar espaço. `stall_timeout_seconds` detecta ausência de progresso do processo, não limita a duração total.

In [ ]:
CFG={
  'dataset':'manu/project_gutenberg',
  'languages':['pt','en','es'],
  'max_unique_per_language':None,
  'max_scan_per_language':None,
  'max_book_bytes':80_000_000,
  'progress_every':100,
  'state_every':25,
  'stall_timeout_seconds':1800,
  'max_attempts_per_language':5,
  'retry_wait_seconds':60,
  'verify_existing_on_resume':False,
  'min_free_drive_bytes':1_000_000_000,
  'enable_auto_rights':True,
  'enable_cleaned_copies':False,
}
CONFIG=BASE/'configs'/'production_v08.json'
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(CFG,ensure_ascii=False,indent=2))

## 3. Ambiente isolado para Hugging Face (não altera bibliotecas do Colab)

In [ ]:
VENV=Path('/content/maia-gutenberg-v08-env'); PY=VENV/'bin/python'
if not PY.exists():
    try:
        subprocess.run([sys.executable,'-m','virtualenv','--version'],check=True,capture_output=True,timeout=20)
    except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
        subprocess.run([sys.executable,'-m','pip','install','-q','virtualenv'],check=True,timeout=180)
    subprocess.run([sys.executable,'-m','virtualenv',str(VENV)],check=True,timeout=120)
subprocess.run([str(PY),'-m','pip','install','-q','datasets>=3,<5','huggingface_hub>=0.30,<2','requests>=2.32,<3'],check=True,timeout=600)
subprocess.run([str(PY),'-c','import datasets; print("datasets",datasets.__version__)'],check=True,timeout=30)

## 4. Instalar worker de aquisição com checkpoint e escrita atômica

In [ ]:
WORKER_SOURCE = "import argparse, collections, datetime, hashlib, json, os, re, sys, time, traceback, shutil\nfrom pathlib import Path\nfrom datasets import load_dataset\n\ndef now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()\ndef atomic_json(path,obj):\n path.parent.mkdir(parents=True,exist_ok=True)\n tmp=path.with_name('.'+path.name+'.'+str(os.getpid())+'.part')\n with tmp.open('w',encoding='utf-8') as f:\n  json.dump(obj,f,ensure_ascii=False,indent=2); f.flush(); os.fsync(f.fileno())\n os.replace(tmp,path)\ndef append_jsonl(path,obj):\n with path.open('a',encoding='utf-8') as f:\n  f.write(json.dumps(obj,ensure_ascii=False)+'\\n'); f.flush(); os.fsync(f.fileno())\ndef main():\n ap=argparse.ArgumentParser()\n for arg in ('root','config','language'): ap.add_argument('--'+arg,required=True)\n a=ap.parse_args(); cfg=json.loads(Path(a.config).read_text()); lang=a.language\n base=Path(a.root)/'gutenberg_v08'; old=Path(a.root)/'gutenberg_v06'\n bookdir=base/'quarantine'/'books'/lang; bookdir.mkdir(parents=True,exist_ok=True)\n manifest=base/'manifests'/f'{lang}.jsonl'; statepath=base/'state'/f'{lang}.json'\n oldmanifest=old/'manifests'/f'{lang}.jsonl'\n ids={}; hashes=set(); old_ids=set(); old_hashes=set()\n def load_manifest(path,is_old):\n  if not path.exists(): return\n  for line in path.open(encoding='utf-8'):\n   try: x=json.loads(line)\n   except Exception: continue\n   if x.get('status')=='saved':\n    sha=x.get('sha256'); bid=str(x.get('book_id') or '')\n    if not isinstance(sha,str) or not re.fullmatch('[0-9a-f]{64}',sha): continue\n    if is_old:\n     if (old/'quarantine'/'books'/lang/(sha+'.txt')).is_file():\n      old_hashes.add(sha)\n      if bid: old_ids.add(bid)\n    else:\n     hashes.add(sha)\n     if bid: ids[bid]=sha\n load_manifest(oldmanifest,True); load_manifest(manifest,False)\n # Recovery of files committed before manifest append; hashes recover, ID unknown.\n recovered=0\n for p in bookdir.glob('*.txt'):\n  sha=p.stem\n  if re.fullmatch('[0-9a-f]{64}',sha) and sha not in hashes:\n   h=hashlib.sha256()\n   with p.open('rb') as f:\n    for block in iter(lambda:f.read(1024*1024),b''): h.update(block)\n   if h.hexdigest()==sha:\n    hashes.add(sha); recovered+=1\n    append_jsonl(manifest,{'status':'saved','book_id':None,'sha256':sha,'bytes':p.stat().st_size,'language':lang,'source':'recovered_orphan','timestamp_utc':now()})\n state={}\n if statepath.exists():\n  try: state=json.loads(statepath.read_text(encoding='utf-8'))\n  except Exception: pass\n cursor=int(state.get('cursor',0))\n if cursor<0: cursor=0\n report={'language':lang,'started_utc':now(),'cursor_start':cursor,'cursor_end':cursor,\n  'scanned_this_run':0,'saved_this_run':0,'duplicate_id':0,'duplicate_sha':0,\n  'skipped_existing_v06':0,'oversize':0,'missing_text':0,'invalid_rows':0,\n  'recovered_orphans':recovered,'error':None,'completed':False}\n def checkpoint(done=False):\n  state.update({'language':lang,'dataset':cfg['dataset'],'cursor':cursor,\n   'complete':done,'last_update_utc':now(),'new_unique':len(hashes),\n   'v06_existing':len(old_hashes)})\n  atomic_json(statepath,state)\n try:\n  print('OPEN',cfg['dataset'],lang,'cursor',cursor,'existing new',len(hashes),'v06',len(old_hashes),flush=True)\n  ds=load_dataset(cfg['dataset'],split=lang,streaming=True)\n  # HF skip may still consume network internally; index remains a reproducible logical cursor.\n  if cursor: ds=ds.skip(cursor)\n  print('READY',lang,flush=True)\n  for row in ds:\n   maxscan=cfg.get('max_scan_per_language')\n   if maxscan is not None and cursor>=int(maxscan):\n    print('SCAN_LIMIT',lang,cursor,flush=True); break\n   maxunique=cfg.get('max_unique_per_language')\n   if maxunique is not None and len(hashes)+len(old_hashes)>=int(maxunique):\n    print('UNIQUE_LIMIT',lang,flush=True); break\n   # Cursor increments only after processing the record.\n   if not isinstance(row,dict):\n    report['invalid_rows']+=1; cursor+=1; report['scanned_this_run']+=1; continue\n   bid=str(row.get('id') or '').strip(); txt=row.get('text')\n   if not isinstance(txt,str) or not txt:\n    report['missing_text']+=1\n   elif bid and (bid in ids or bid in old_ids):\n    report['duplicate_id']+=1\n   else:\n    raw=txt.encode('utf-8')\n    if len(raw)>int(cfg['max_book_bytes']):\n     report['oversize']+=1\n     append_jsonl(manifest,{'status':'skipped_oversize','book_id':bid,'bytes':len(raw),'language':lang,'timestamp_utc':now()})\n    else:\n     sha=hashlib.sha256(raw).hexdigest()\n     if sha in hashes or sha in old_hashes:\n      report['duplicate_sha']+=1\n      append_jsonl(manifest,{'status':'duplicate_sha','book_id':bid,'sha256':sha,'language':lang,'timestamp_utc':now()})\n      if bid: ids[bid]=sha\n     else:\n      # A low-free-space guard is advisory only; Drive quota may not be reflected.\n      try: free=shutil.disk_usage(bookdir).free\n      except Exception: free=10**18\n      if free<int(cfg.get('min_free_drive_bytes',0)):\n       raise OSError('Insufficient available mounted filesystem space; free storage then rerun')\n      path=bookdir/(sha+'.txt'); tmp=bookdir/('.'+sha+'.'+str(os.getpid())+'.part')\n      try:\n       with tmp.open('wb') as f: f.write(raw); f.flush(); os.fsync(f.fileno())\n       os.replace(tmp,path)\n      finally:\n       if tmp.exists(): tmp.unlink()\n      append_jsonl(manifest,{'status':'saved','book_id':bid or None,'sha256':sha,\n       'bytes':len(raw),'characters':len(txt),'language':lang,\n       'source_dataset':cfg['dataset'],'source_split':lang,\n       'timestamp_utc':now(),'review_status':'quarantine_automated_rights_pending'})\n      hashes.add(sha)\n      if bid: ids[bid]=sha\n      report['saved_this_run']+=1\n   cursor+=1; report['scanned_this_run']+=1\n   if report['scanned_this_run']%int(cfg['state_every'])==0: checkpoint()\n   if report['scanned_this_run']%int(cfg['progress_every'])==0:\n    print('PROGRESS',lang,'cursor',cursor,'new_unique',len(hashes),'v06',len(old_hashes),\n      'saved_run',report['saved_this_run'],'dup_id',report['duplicate_id'],flush=True)\n  else:\n   report['completed']=True\n  checkpoint(done=report['completed'])\n except BaseException as exc:\n  report['error']=type(exc).__name__+': '+str(exc)\n  checkpoint(done=False)\n  print('ERROR',lang,report['error'],flush=True)\n  traceback.print_exc()\n finally:\n  report['cursor_end']=cursor; report['new_unique_total']=len(hashes)\n  report['existing_v06_unique']=len(old_hashes); report['finished_utc']=now()\n  stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n  rp=base/'reports'/f'acquire_{lang}_{stamp}.json'; atomic_json(rp,report)\n  print('FINAL',json.dumps(report,ensure_ascii=False),flush=True)\n if report['error']: sys.exit(2)\nif __name__=='__main__': main()\n"
WORKER_PATH=BASE/'workers'/'acquire_v08.py'
WORKER_PATH.write_text(WORKER_SOURCE,encoding='utf-8')
import ast
ast.parse(WORKER_SOURCE)
print('Worker pronto:',WORKER_PATH)

## 5. Executar aquisição completa (retomável)

A execução pode levar muitas horas, dependendo da rede e do tamanho real das partições. **Não há limite de 600 s por idioma**. O supervisor interrompe somente se não houver saída por 30 minutos; cada tentativa retoma do último cursor. Ao reiniciar o Colab, execute novamente as células 1–5. Se uma partição terminar, o estado `complete=true` evita reprocessamento.

In [ ]:
def run_one(lang,attempt):
    logfile=BASE/'logs'/f'acquire_{lang}_{datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")}_attempt{attempt}.log'
    cmd=[str(PY),'-u',str(WORKER_PATH),'--root',str(ROOT),'--config',str(CONFIG),'--language',lang]
    with logfile.open('w',encoding='utf-8') as log:
        proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        pos=0; last_output=time.monotonic(); timed_out=False
        try:
            while proc.poll() is None:
                with logfile.open(encoding='utf-8',errors='replace') as f:
                    f.seek(pos); chunk=f.read(); pos=f.tell()
                if chunk:
                    print(chunk,end='',flush=True); last_output=time.monotonic()
                if time.monotonic()-last_output>CFG['stall_timeout_seconds']:
                    timed_out=True; print('STALL_TIMEOUT',lang,'tentativa',attempt,flush=True)
                    os.killpg(proc.pid,signal.SIGTERM)
                    try: proc.wait(timeout=10)
                    except subprocess.TimeoutExpired: os.killpg(proc.pid,signal.SIGKILL); proc.wait()
                    break
                time.sleep(2)
        except KeyboardInterrupt:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
                try: proc.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(proc.pid,signal.SIGKILL); proc.wait()
            raise
        with logfile.open(encoding='utf-8',errors='replace') as f:
            f.seek(pos); print(f.read(),end='',flush=True)
    return {'language':lang,'attempt':attempt,'returncode':proc.returncode,'stalled':timed_out,'log':str(logfile)}

RUNS=[]
for lang in CFG['languages']:
    statepath=BASE/'state'/f'{lang}.json'
    if statepath.exists() and json.loads(statepath.read_text()).get('complete'):
        print('ALREADY_COMPLETE',lang); continue
    for attempt in range(1,CFG['max_attempts_per_language']+1):
        print('===',lang,'attempt',attempt,'===',flush=True)
        result=run_one(lang,attempt); RUNS.append(result)
        state=json.loads(statepath.read_text()) if statepath.exists() else {}
        if state.get('complete'):
            print('COMPLETE',lang); break
        if attempt<CFG['max_attempts_per_language']:
            print('Tentativa interrompida. Retomada em',CFG['retry_wait_seconds'],'segundos.',flush=True)
            time.sleep(CFG['retry_wait_seconds'])
    else:
        print('ATENÇÃO:',lang,'não concluiu. Reexecute esta célula para continuar.',flush=True)
print('Resumo:',json.dumps(RUNS[-10:],ensure_ascii=False,indent=2))

## 6. Metadados de direitos — catálogo oficial do Project Gutenberg

Baixa uma vez o catálogo CSV oficial e mantém cópia no Drive. Falhas de rede não impedem a aquisição nem o inventário: resultam em **quarentena automática**. O catálogo é útil para identificar autores e títulos, mas **não comprova por si só a situação de traduções, edições ou direitos de treinamento**. A política padrão não aprova automaticamente apenas pela morte do autor.

In [ ]:
import csv, re, hashlib, urllib.request, urllib.error
CATALOG=BASE/'rights'/'pg_catalog.csv'
CATALOG_URL='https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv'
if not CATALOG.exists():
    print('Baixando catálogo Gutenberg...',flush=True)
    try:
        req=urllib.request.Request(CATALOG_URL,headers={'User-Agent':'MaiaCorpusResearch/0.8 (metadata audit)'})
        with urllib.request.urlopen(req,timeout=90) as response:
            tmp=CATALOG.with_suffix('.csv.part')
            with tmp.open('wb') as f:
                while True:
                    chunk=response.read(1024*1024)
                    if not chunk: break
                    f.write(chunk)
            os.replace(tmp,CATALOG)
    except Exception as exc:
        print('Catálogo indisponível:',type(exc).__name__,str(exc))
print('Catálogo presente:',CATALOG.exists(), 'bytes:',CATALOG.stat().st_size if CATALOG.exists() else 0)

## 7. Auditoria automática de direitos e limpeza conservadora

**Decisão padrão:** `AUTO_QUARANTINE`. `AUTO_REJECTED` se houver evidência explícita de restrição incompatível; `AUTO_APPROVED` apenas com evidência positiva **para a versão textual exata**, que o catálogo comum não fornece. O campo `author_death_year` é uma pista para triagem, não autorização. A regra brasileira de 70 anos pode ser insuficiente para traduções, coautoria, edições e outros direitos.

A limpeza textual fica **desabilitada por padrão**. Ao ativar `enable_cleaned_copies`, somente remove blocos quando encontrar delimitadores Gutenberg inequívocos; casos ambíguos preservam texto integral e recebem flag. Cópias derivadas também ficam em quarentena.

In [ ]:
RIGHTS_SOURCE = "import csv, hashlib, json, re, datetime, os, collections, traceback\nfrom pathlib import Path\nroot=Path(__import__('sys').argv[1]); cfg=json.loads((root/'gutenberg_v08/configs/production_v08.json').read_text())\nbase=root/'gutenberg_v08'; old=root/'gutenberg_v06'\nrights=base/'rights'; rights.mkdir(exist_ok=True)\nnow=datetime.datetime.now(datetime.timezone.utc).isoformat()\n# Catalog columns often: Text#, Type, Issued, Title, Language, Authors, Subjects, LoCC, Bookshelves\ncatalog={}; catalog_error=None\ncp=rights/'pg_catalog.csv'\nif cp.exists():\n try:\n  with cp.open(encoding='utf-8-sig',newline='') as f:\n   for r in csv.DictReader(f):\n    key=str(r.get('Text#') or r.get('Text #') or r.get('Text') or '').strip()\n    if key: catalog[key]=r\n except Exception as e: catalog_error=str(e)\nelse: catalog_error='catalog_missing'\ndef get_meta(bid):\n match=re.match(r'^(\\d+)',str(bid or ''))\n return catalog.get(match.group(1),{}) if match else {}\ndef manifest_rows(folder,lang):\n p=folder/'manifests'/f'{lang}.jsonl'\n if not p.exists(): return []\n rows=[]\n for line in p.open(encoding='utf-8'):\n  try:\n   x=json.loads(line)\n   if x.get('status')=='saved' and x.get('sha256'): rows.append(x)\n  except Exception: pass\n return rows\nstart_re=re.compile(r'\\*\\*\\*\\s*START OF (?:THE|THIS) PROJECT GUTENBERG[^\\n]*\\*\\*\\*',re.I)\nend_re=re.compile(r'\\*\\*\\*\\s*END OF (?:THE|THIS) PROJECT GUTENBERG[^\\n]*\\*\\*\\*',re.I)\ndef clean(text):\n a=start_re.search(text[:20000]); z=end_re.search(text[-20000:])\n if a and z:\n  endpos=len(text)-20000+z.start() if len(text)>20000 else z.start()\n  if endpos>a.end(): return text[a.end():endpos].strip(), 'both_delimiters_removed'\n return text,'ambiguous_preserved'\nrows=[]; seen=set(); problems=[]\nfor lang in ('pt','en','es'):\n for source,folder in (('v06',old),('v08',base)):\n  for x in manifest_rows(folder,lang):\n   sha=x['sha256']; key=(lang,sha)\n   if key in seen: continue\n   seen.add(key)\n   path=folder/'quarantine'/'books'/lang/(sha+'.txt')\n   meta=get_meta(x.get('book_id'))\n   rec={'language':lang,'source':source,'book_id':x.get('book_id') or '',\n     'sha256':sha,'path':str(path),'title':meta.get('Title',''),\n     'authors':meta.get('Authors',''),'catalog_language':meta.get('Language',''),\n     'catalog_type':meta.get('Type',''),'catalog_issued':meta.get('Issued',''),\n     'catalog_url':('https://www.gutenberg.org/ebooks/'+str(x['book_id']).split('-')[0]) if x.get('book_id') else '',\n     'source_dataset':'manu/project_gutenberg','jurisdiction':'BR',\n     'rights_status':'AUTO_QUARANTINE','rights_reason':'No verified rights grant for exact edition/translation and training use',\n     'rights_evidence':'metadata_only' if meta else 'metadata_unavailable',\n     'approved_for_training':False,'integrity_ok':False,\n     'original_bytes':None,'original_characters':None,\n     'start_marker':False,'end_marker':False,'cleaning_status':'not_requested',\n     'cleaned_sha256':'','cleaned_characters':None}\n   try:\n    raw=path.read_bytes(); digest=hashlib.sha256(raw).hexdigest()\n    txt=raw.decode('utf-8','strict')\n    rec['integrity_ok']=digest==sha\n    rec['original_bytes']=len(raw); rec['original_characters']=len(txt)\n    rec['start_marker']=bool(start_re.search(txt[:20000]))\n    rec['end_marker']=bool(end_re.search(txt[-20000:]))\n    if not rec['integrity_ok']:\n     rec['rights_reason']='Integrity failure; quarantined'; problems.append({'path':str(path),'issue':'sha_mismatch'})\n    if cfg.get('enable_cleaned_copies') and rec['integrity_ok']:\n     cleaned,status=clean(txt); rec['cleaning_status']=status\n     out=base/'cleaned_quarantine'/lang; out.mkdir(parents=True,exist_ok=True)\n     data=cleaned.encode('utf-8'); clean_sha=hashlib.sha256(data).hexdigest()\n     target=out/(clean_sha+'.txt')\n     if not target.exists():\n      tmp=out/('.'+clean_sha+'.part')\n      with tmp.open('wb') as f: f.write(data); f.flush(); os.fsync(f.fileno())\n      os.replace(tmp,target)\n     rec['cleaned_sha256']=clean_sha; rec['cleaned_characters']=len(cleaned)\n   except Exception as exc:\n    problems.append({'path':str(path),'issue':type(exc).__name__+': '+str(exc)})\n    rec['rights_reason']='Unreadable or missing source; quarantined'\n   rows.append(rec)\n print('AUDIT',lang,'records',sum(r['language']==lang for r in rows),flush=True)\nfields=list(rows[0]) if rows else ['language']\ninv=rights/'rights_inventory_latest.csv'; tmp=rights/'rights_inventory_latest.csv.part'\nwith tmp.open('w',encoding='utf-8',newline='') as f:\n w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)\nos.replace(tmp,inv)\nsummary={'timestamp_utc':now,'source_dataset':'manu/project_gutenberg',\n 'catalog_available':bool(catalog),'catalog_records':len(catalog),'catalog_error':catalog_error,\n 'books_total':len(rows),'books_by_language':dict(collections.Counter(r['language'] for r in rows)),\n 'metadata_matched':sum(bool(r['title']) for r in rows),\n 'integrity_ok':sum(r['integrity_ok'] for r in rows),\n 'start_markers':sum(r['start_marker'] for r in rows),\n 'end_markers':sum(r['end_marker'] for r in rows),\n 'auto_approved':0,'auto_quarantine':len(rows),'auto_rejected':0,\n 'problems':problems,\n 'policy':'Fail closed: no training approval from catalog metadata or Gutenberg availability alone; exact edition and translation rights unverified.'}\nsp=rights/'rights_summary_latest.json'; temp=sp.with_suffix('.json.part')\ntemp.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8'); os.replace(temp,sp)\nprint('SUMMARY',json.dumps({k:v for k,v in summary.items() if k!='problems'},ensure_ascii=False,indent=2),flush=True)\nprint('INVENTORY',inv,flush=True)\n"
RIGHTS_PATH=BASE/'workers'/'rights_v08.py'
RIGHTS_PATH.write_text(RIGHTS_SOURCE,encoding='utf-8')
ast.parse(RIGHTS_SOURCE)
print('Auditoria pronta:',RIGHTS_PATH)

In [ ]:
# Pode executar enquanto a aquisição estiver pausada, ou após ela terminar.
# Reexecutar atualiza o inventário com novos livros, sem apagar os originais.
result=subprocess.run([sys.executable,'-u',str(RIGHTS_PATH),str(ROOT)],check=False)
print('Exit code:',result.returncode)
assert result.returncode==0, 'Consulte a saída acima; os arquivos originais continuam intactos.'

## 8. Painel de progresso e próximos passos

In [ ]:
for lang in CFG['languages']:
    p=BASE/'state'/f'{lang}.json'
    state=json.loads(p.read_text()) if p.exists() else {}
    print(lang,'cursor=',state.get('cursor',0),'novos=',state.get('new_unique',0),
          'v06=',state.get('v06_existing',0),'completo=',state.get('complete',False),
          'atualizado=',state.get('last_update_utc','—'))
s=BASE/'rights'/'rights_summary_latest.json'
if s.exists():
    d=json.loads(s.read_text()); print('Auditoria:',d['books_total'],'livros,',
      d['metadata_matched'],'metadados identificados,',d['auto_approved'],'aprovados,',
      d['auto_quarantine'],'em quarentena')
print('Relatórios:',BASE/'reports')
print('Inventário de direitos:',BASE/'rights'/'rights_inventory_latest.csv')

### Decisões e limites explícitos

1. **Sem aprovação jurídica fictícia:** o catálogo Gutenberg informa metadados bibliográficos, não concede licença para treinamento da edição exata. Por isso o classificador padrão é conservador. Para aprovações automáticas reais, uma versão posterior deve integrar fontes verificáveis de licença por item, direitos de traduções e política jurídica revisada.
2. **Sem garantia de download completo em uma sessão:** Colab pode desconectar e o Google Drive pode ficar sem cota. O estado persistente permite retomar; o notebook não promete contornar limites de armazenamento ou disponibilidade do Hub.
3. **Escopo:** todas as linhas disponíveis das partições `pt`, `en` e `es`, não o dataset inteiro em outros idiomas. Livros acima de `max_book_bytes` são registrados como omitidos, nunca truncados.
4. **Custos:** muitos arquivos pequenos no Drive são mais lentos que arquivos grandes; esta versão privilegia retomada e integridade. Uma etapa posterior pode empacotar em shards sem alterar os hashes originais.
5. **Limpeza:** cópias derivadas podem ser ativadas depois de observar a cobertura dos marcadores. O conteúdo original permanece imutável.